# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step data exploration and analysis for the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}")
print(f"License: {md.license}")
print(f"Keywords: {md.keywords}")

## 2. Data Overview
Review available record sets, their IDs, and fields. Use these `@id` values for further data extraction and processing. All identifiers reference the Croissant schema definitions.

We'll enumerate the available record sets and their fields.

In [ ]:
# Get all record set metadata
from collections.abc import Sequence

# Helper to ensure always a list
def to_list(x):
    if isinstance(x, Sequence) and not isinstance(x, str):
        return list(x)
    elif x is None:
        return []
    else:
        return [x]

record_sets = dataset.metadata.recordSet
if not isinstance(record_sets, list):
    record_sets = [record_sets]
print("Record sets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id'] if isinstance(rs, dict) else rs}")

# Show fields for each record set
print("\nRecord sets and their fields:")
if len(record_sets) == 0:
    print("No direct record sets in top-level Croissant metadata; trying schema:Dataset parts.")
    # Fallback: try dataset.distribution
    for dist in to_list(dataset.metadata.distribution):
        print(f"Distribution: {dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist}")
        dist_obj = None
        # Try loading as a record set
        try:
            df = pd.DataFrame(dataset.records(record_set=dist['@id']))
            print(f"  -> Columns: {df.columns.tolist()}")
        except Exception as e:
            print(f"  Could not load as a record set: {e}")
else:
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        print(f"Record set @id: {rs_id}")
        # Show the field IDs
        fields = rs.get('field', []) if isinstance(rs, dict) and 'field' in rs else []
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            for field in fields:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
                print(f"  Field @id: {field_id}")
        else:
            print("  No field listed in metadata.")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis.

**Note:** All references use the record set and field `@id`s as discovered in the overview steps.


In [ ]:
# Manually specify the record set(s) identified above for this dataset

# As of 2024-06, FAIR^2 schema often encodes its record sets in `distribution`. We'll extract those, and look for their usage as record sets.

record_sets = []
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    for rs in to_list(dataset.metadata.recordSet):
        if isinstance(rs, dict) and '@id' in rs:
            record_sets.append(rs['@id'])
        else:
            record_sets.append(rs)
else:
    # Fallback to `distribution`, which may act as record set
    for dist in to_list(dataset.metadata.distribution):
        if isinstance(dist, dict) and '@id' in dist:
            record_sets.append(dist['@id'])
        else:
            record_sets.append(dist)

dataframes = {}
for rs_id in record_sets:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"Loaded record set {rs_id} (columns: {df.columns.tolist()[:6]} ...)")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Show columns of first record set loaded
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst dataframe columns (@id): {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("Did not load any valid record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps: filtering, normalization, grouping, or other transformations for analysis.

**Note:**
- All fields are referenced by their `@id` as specified in schema/columns.
- You may need to inspect the above-printed columns to select appropriate fields.

In [ ]:
# For demonstration, we use the first available dataframe and select example numeric fields.

# STEP 1: List numeric fields for main record set
import numpy as np

df = dataframes[main_rs_id]

# Try and guess numeric columns
numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if not numeric_fields:
    # Try to coerce columns to numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c], errors='ignore')
        except Exception:
            pass
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

print(f"Numeric field candidates (@id): {numeric_fields}")

# Use the first numeric field by @id
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # e.g., 'cr:peptide_count' or similar
    threshold = np.nanmean(df[numeric_field_id])
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a grouping field (e.g. 'cr:modification' or any categorical with <= 20 unique values)
    candidates = [c for c in df.columns if df[c].nunique() < 20 and c != numeric_field_id]
    if candidates:
        group_field_id = candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df)
    else:
        print("No suitable grouping field found.")
else:
    print("No numeric fields found to demonstrate EDA. Please inspect the dataframe above for field names.")

## 5. Visualization
Visualize data distributions or relationships between fields for deeper understanding.


In [ ]:
# If you have matplotlib, seaborn, or plotly, you can visualize numeric columns:
import matplotlib.pyplot as plt

if numeric_fields:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30, color='teal')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping in previous step
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        plt.barh(grouped_df[group_field_id].astype(str), grouped_df[numeric_field_id])
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 Croissant dataset, reviewed its structure, extracted and processed data by referencing all fields via their `@id`, and performed early exploratory analysis and visualizations. All downstream field, record set, and column references have been by their Croissant `@id`.

Further analysis can include statistical testing, advanced transformations, or model building using the referenced DataFrame.
